In [ ]:
import numpy as np
import mtd
import matplotlib.pyplot as plt

import scipy.stats as st
import statsmodels.api as sm
import statsmodels.stats.multitest as mt
from tqdm import tqdm

import seaborn as sns

sns.set_theme(style="darkgrid")
FIG_SIZE = (18,5)
sns.set_context("talk")

import torch
import torchvision
import torchvision.transforms as transforms
from PIL import Image as im
from matplotlib import gridspec

In [ ]:
import pandas as pd

In [ ]:
from utils import calculate_K_Hs

In [ ]:
def K_H(h, kernel, u):
    return  kernel(u/h)/(h**2)

In [ ]:
def kernel_den_estimator(
    diagrams,
    kernel,
    bounds = np.array([[0.0, 0.0], [1.0, 1.0]]),
    resolution = 64,
    h = 0.1,
):
    n = len(diagrams)
    
    x = np.linspace(bounds[1, 0], bounds[0, 0], resolution)
    y = np.linspace(bounds[1, 1], bounds[0, 1], resolution)
    X,Y = np.meshgrid(x,y)

    result = np.zeros((resolution, resolution))

    for i in tqdm(range(resolution)):
        for j in range(resolution):
            u = np.array([X[i, j], Y[i, j]])
            for diag in diagrams:
                result[i, j] = np.sum(K_H(h, kernel, u - diag))/len(diag)
            result[i, j] = result[i, j]/len(diagrams)
    return X, Y, result

In [ ]:
def persistence_surface(dgms, h , kernel, w, bounds, resolution = 64):
    dgm = np.vstack(dgms)
    N = len(dgms)
    
    x = np.linspace(bounds[1, 0], bounds[0, 0], resolution)
    y = np.linspace(bounds[1, 1], bounds[0, 1], resolution)
    X,Y = np.meshgrid(x,y)

    result = np.zeros((resolution, resolution))

    for i in tqdm(range(resolution)):
        for j in range(resolution):
            
            u = np.array([X[i, j], Y[i, j]])
            result[i, j] = np.sum(K_H(h, kernel, u-dgm) * w(dgm))
            
    result = result/N

    return X, Y, result

In [ ]:
class AddGaussianNoise(object):
    def __init__(self, mean=0., std=1.):
        self.std = std
        self.mean = mean
        
    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean
    
    def __repr__(self):
        return self.__class__.__name__ + '(mean={0}, std={1})'.format(self.mean, self.std)

In [ ]:
import pickle
import datetime

def calculate_barcs_with_noise(clouds, clouds_noised, classes, dataset_name="", batch_size1=1000, batch_size2=2000, n_boostrap=100, augs = []):

    
    barcs_of_clouds_clear = {}
    for i in tqdm(range(len(classes)), desc = f"Calculating barcodes of clear clouds"):
        for j in range(len(classes)):
            barcs = [mtd.calc_cross_barcodes(clouds[i], clouds[j], batch_size1 = batch_size1, batch_size2 = batch_size2, pdist_device = "cuda",is_plot = False) for _ in range(n_boostrap)]
            barcs_of_clouds_clear[f"{classes[i]} vs {classes[j]}"] = barcs
            # print(f"{classes[i]} vs {classes[j]} is done !!!")
    with open(f"density_exp/barcs_of_clouds_clear{dataset_name}", "wb") as fp:   #Pickling
        pickle.dump(barcs_of_clouds_clear, fp)

    
    with open('progress_file.txt',"w", encoding="utf-8") as f:    
        now = datetime.datetime.now()
        print(f"Calculated clear clouds at: {now}!",file=f)

    for idx in range(len(augs)):
        str_repr = str(augs[idx])
        barcs_of_clouds_pure_noised = {}
        barcs_of_clouds_pure_noised["noise_name"] = str_repr
        for i in tqdm(range(len(classes)), desc = f"Calculating barcodes of pure and noised ({str_repr}) clouds"):
            for j in range(len(classes)):
                barcs = [mtd.calc_cross_barcodes(clouds[i], clouds_noised[idx][j], batch_size1 = batch_size1, batch_size2 = batch_size2, pdist_device = "cuda",is_plot = False) for _ in range(n_boostrap)]
                barcs_of_clouds_pure_noised[f"{classes[i]} vs {classes[j]}"] = barcs
                # print(f"{classes[i]} vs {classes[j]} is done !!!")
        with open(f"density_exp/barcs_of_clouds_pure_noised_{idx}{dataset_name}", "wb") as fp:   #Pickling
            pickle.dump(barcs_of_clouds_pure_noised, fp)

    
    with open('progress_file.txt', "a", encoding="utf-8") as f:  
        now = datetime.datetime.now()
        print(f"Calculated pure and noised clouds at: {now}!",file=f)


    for idx in range(len(augs)):
        str_repr = str(augs[idx])
        barcs_of_clouds_noised_noised = {}
        barcs_of_clouds_noised_noised["noise_name"] = str_repr
        for i in tqdm(range(len(classes)), desc = f"Calculating barcodes of noised and noised ({str_repr}) clouds"):
            for j in range(len(classes)):
                barcs = [mtd.calc_cross_barcodes(clouds_noised[idx][i], clouds_noised[idx][j], batch_size1 = batch_size1, batch_size2 = batch_size2, pdist_device = "cuda",is_plot = False) for _ in range(n_boostrap)]
                barcs_of_clouds_noised_noised[f"{classes[i]} vs {classes[j]}"] = barcs
                # print(f"{classes[i]} vs {classes[j]} is done !!!")
        with open(f"density_exp/barcs_of_clouds_noised_noised_{idx}{dataset_name}", "wb") as fp:   #Pickling
            pickle.dump(barcs_of_clouds_noised_noised, fp)

    with open('progress_file.txt', "a", encoding="utf-8") as f:  
        now = datetime.datetime.now()
        print(f"Calculated noised and noised clouds at: {now}!",file=f)


####

def calculate_mtd_with_noise(dataset_name="", augs = []):

    mtd_of_clouds_clear = {}
    mtd_of_clouds_pure_noised = [{}, {}, {}]
    mtd_of_clouds_noised_noised = [{}, {}, {}]

    with open(f"density_exp/barcs_of_clouds_clear{dataset_name}", "rb") as fp:   #Pickling
        barcs_of_clouds_clear = pickle.load(fp)
        
    for key, value in barcs_of_clouds_clear.items():
        mtd_of_clouds_clear[key] = [mtd.get_score(barc, 1, 'sum_length') for barc in value]
        
    for idx in range(len(augs)):
        with open(f"density_exp/barcs_of_clouds_pure_noised_{idx}{dataset_name}", "rb") as fp:   #Pickling
            barcs_of_clouds_pure_noised = pickle.load(fp)
            
        for key, value in barcs_of_clouds_pure_noised.items():
            if key!="noise_name":
                mtd_of_clouds_pure_noised[idx][key] = [mtd.get_score(barc, 1, 'sum_length') for barc in value]

    for idx in range(len(augs)):
        with open(f"density_exp/barcs_of_clouds_noised_noised_{idx}{dataset_name}", "rb") as fp:   #Pickling
            barcs_of_clouds_noised_noised = pickle.load(fp)
            
        for key, value in barcs_of_clouds_noised_noised.items():
            if key!="noise_name":
                mtd_of_clouds_noised_noised[idx][key] = [mtd.get_score(barc, 1, 'sum_length') for barc in value]

    return mtd_of_clouds_clear, mtd_of_clouds_pure_noised, mtd_of_clouds_noised_noised

In [ ]:
kernel = lambda x: np.exp(-np.sum(x**2,axis = 1)/2)/(2*np.pi)

In [ ]:
with open("Data/cross_ripsnet_text_exp/human_gpt3_davinci_003_pds_10000_50_boostrap", 'rb') as fp:   #Pickling
    train_PD = pickle.load(fp)

In [ ]:
train_PD_1 = [[x[1] if len(x[1])>0 else np.array([[0., 0.]]) for x in barcs ] for barcs in train_PD]

In [ ]:
len(train_PD_1[0])

In [ ]:
res = calculate_K_Hs(train_PD_1[0], kernel)

In [ ]:
res[0, 0]

In [ ]:
ROOT = "PersistenceUniversality/pd_example_files/"
np.random.seed(7)


P = np.loadtxt(f'{ROOT}annulus.csv')
Q = np.loadtxt(f'{ROOT}annulus1.csv') + np.array([1,0])

In [ ]:
trials = 100

np.random.seed(7)
barcs = [mtd.calc_cross_barcodes(P, Q, batch_size1 = 100, batch_size2 = 400, pdist_device = "cuda", is_plot=False) for _ in range(trials)]

In [ ]:
barcs = np.array(barcs)
barcs = barcs[:, 1]

In [ ]:
## transformation
for pers_dgm in barcs:
    pers_dgm[:, 1] = pers_dgm[:, 1] - pers_dgm[:, 0]

In [ ]:
y_max = np.max(np.vstack(barcs)[:,1])
y_max

In [ ]:
x_max = np.max(np.vstack(barcs)[:,0])
x_max

In [ ]:
w = lambda x: x[:,1]**3
bounds = np.array([[0, 0],[x_max, y_max]])

In [ ]:
X,Y,surface = kernel_den_estimator(barcs, kernel, bounds = np.array([[0, 0],[x_max, y_max]]))

In [ ]:
plt.pcolormesh(X,Y,surface)

In [ ]:
b = np.vstack(barcs)

In [ ]:
plt.scatter(b[:,0],b[:,1])

In [ ]:
X,Y,Z = persistence_surface(barcs,0.1,kernel,w,bounds)

In [ ]:
plt.pcolormesh(X,Y,Z)

In [ ]:
fig, ax = plt.subplots() 
ax.imshow(Z, cmap='jet')
plt.xticks([])
plt.yticks([])

In [ ]:
from gudhi.representations import DiagramSelector
pds_train      = DiagramSelector(use=True).fit_transform(barcs)

In [ ]:
from gudhi.representations import  PersistenceImage
PI_params = {'bandwidth': 0.1, 'weight': lambda x: x[1]**3, 
             'resolution': [50,50]}
PI_train = PersistenceImage(**PI_params).fit_transform(pds_train)
MPI = np.max(PI_train)
PI_train /= MPI

In [ ]:
PI_train = np.sum(PI_train,axis = 0)/100

In [ ]:
fig, ax = plt.subplots() 
ax.imshow(np.reshape(PI_train, [50,50]), cmap='jet')
plt.xticks([])
plt.yticks([])

This function makes birthpersistence transformation inside!!!

---

### Let's try to estimate the density of MTD on MNIST

In [ ]:
augs = [AddGaussianNoise(0, 0.5), AddGaussianNoise(0, 0.7), AddGaussianNoise(0, 1.5)]

In [ ]:
from sklearn.datasets import fetch_openml
from PIL import Image

# Load data from https://www.openml.org/d/554
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

In [ ]:
classes = ["ones", "twos", "threes", "fours", "fives", "sixes", "sevens", "eights", "nines"]

In [ ]:
#Create point clouds of "number"s 
def get_dataset(number, transform =None):
    images = []

    for i, elem in enumerate(y):
        if elem == number:
            A = np.zeros((40, 40))
            A[6:34, 6:34] = X[i].reshape((28, 28))
            if transform:
                A = transform(A)
            img = Image.fromarray(np.uint8(A).squeeze(), 'L')
            
            images.append(np.asarray(img).flatten())
    
    return np.array(images)

In [ ]:
clouds = []
for number in ["1", "2", "3", "4", "5", "6", "7", "8", "9"]:
    clouds.append(get_dataset(number))

In [ ]:
transformations = [transforms.Compose(
[transforms.ToTensor(), augmentation]) for augmentation in augs]

# transform = transforms.Compose(
#     [transforms.ToTensor(), AddGaussianNoise(0, 0.7)])
clouds_noised = []
for transform in transformations:
    clouds_noised.append([])
    for number in ["1", "2", "3", "4", "5", "6", "7", "8", "9"]:
        clouds_noised[-1].append(get_dataset(number, transform))

In [ ]:
cloud_5 = get_dataset("5")
cloud_5.shape

In [ ]:
img = Image.fromarray(clouds[0][0].reshape((40,40)))

img.show()

for clouds_noised_aug in clouds_noised:
    img = Image.fromarray(clouds_noised_aug[0][0].squeeze().reshape((40,40)))
    
    img.show()

---

In [ ]:
calculate_barcs_with_noise(clouds, clouds_noised, classes, "_MNIST", augs = augs, n_boostrap=100, batch_size1=1000, batch_size2=2000)


---

#### Let's reproduce experiments for CIFAR10 on MNIST.

In [ ]:
mtd_of_clouds_clear, mtd_of_clouds_pure_noised, mtd_of_clouds_noised_noised = calculate_mtd_with_noise("_MNIST", augs = augs)

In [ ]:
idx = 1
title = str(augs[idx])


i = -1
for main_class in classes:
    i += 1
    f, axs = plt.subplots(1, 3, figsize=(24, 6))
    f.suptitle(f"Noised by {title}", fontsize=16)
    key = f"{main_class} vs {main_class}"
    sns.kdeplot(mtd_of_clouds_clear[key], label=f"{main_class}", ax = axs[0], linestyle="--")
    sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=f"{main_class}", ax = axs[1], linestyle="--")
    sns.kdeplot(mtd_of_clouds_noised_noised[idx][key], label=f"{main_class}", ax = axs[2], linestyle="--")
    for second_class in classes:
        if second_class != main_class:
            key = f"{main_class} vs {second_class}"
            sns.kdeplot(mtd_of_clouds_clear[key], label=key, ax = axs[0])
            sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=key, ax = axs[1])
            sns.kdeplot(mtd_of_clouds_noised_noised[idx][key], label=key, ax = axs[2])

            axs[0].set_title('Pure-Pure')
            axs[1].set_title('Pure-Noise')
            axs[2].set_title('Noise-Noise')
            
    
    plt.legend(fontsize = "xx-small")

In [ ]:
i = -1
for main_class in classes:
    i += 1
    f, axs = plt.subplots(1, 4, figsize=(24, 6))
    f.suptitle(f"Comparing Pure-Noised with different degree", fontsize=16)
    key = f"{main_class} vs {main_class}"
    sns.kdeplot(mtd_of_clouds_clear[key], label=f"{main_class}", ax = axs[0], linestyle="--")
    for idx in range(len(augs)):
        sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=f"{main_class}", ax = axs[idx+1], linestyle="--")
        
    for second_class in classes:
        if second_class != main_class:
            key = f"{main_class} vs {second_class}"

            sns.kdeplot(mtd_of_clouds_clear[key], label=key, ax = axs[0])
            axs[0].set_title(f"Pure")
            
            for idx in range(len(augs)):
                sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=key, ax = axs[idx+1])
                title = str(augs[idx])
                axs[idx+1].set_title(f"Noised by {title}", fontsize=10)
            
    
    plt.legend(fontsize = "xx-small")

---

### Let's try to estimate the density of MTD on CIFAR10

In [ ]:
augs = [AddGaussianNoise(0, 0.05), AddGaussianNoise(0, 0.1), AddGaussianNoise(0, 0.2)]

transformations = [transforms.Compose(
[transforms.ToTensor(), augmentation]) for augmentation in augs]

trainset = torchvision.datasets.CIFAR10(root='./data_cifar10', train=True,
                                        download=True, transform=transforms.ToTensor())


trainset_noised = [torchvision.datasets.CIFAR10(root='./data_cifar10', train=True,
                                        download=True, transform=trans) for trans in transformations]


In [ ]:
len(trainset)

In [ ]:
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

imshow(trainset[0][0])
imshow(trainset_noised[0][0][0])
imshow(trainset_noised[1][0][0])
imshow(trainset_noised[2][0][0])

In [ ]:
classes = trainset.classes

In [ ]:
trainset.class_to_idx

In [ ]:
batch_size = 50000

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

dataiter = iter(trainloader)
images, labels = next(dataiter)

images = images.numpy().mean(axis = 1)

images = images.reshape(50000,-1)

images.shape

In [ ]:
clouds = []
for i in range(10):
    clouds.append(images[labels==i])

---

In [ ]:
batch_size = 50000
clouds_noised = [[], [], []]

for i in range(len(trainset_noised)):
    trainloader = torch.utils.data.DataLoader(trainset_noised[i], batch_size=batch_size,
                                              shuffle=True, num_workers=1)

    dataiter = iter(trainloader)
    images, labels = next(dataiter)
    
    images = images.numpy().mean(axis = 1)
    
    images = images.reshape(50000,-1)
    
    for j in range(10):
        clouds_noised[i].append(images[labels==j])

images.shape

In [ ]:
clouds_noised[0][0].shape

---

In [ ]:
calculate_barcs_with_noise(clouds, clouds_noised, classes, "_CIFAR10", augs = augs, n_boostrap=100, batch_size1=1000, batch_size2=2000)


In [ ]:
mtd_of_clouds_clear, mtd_of_clouds_pure_noised, mtd_of_clouds_noised_noised = calculate_mtd_with_noise("_CIFAR10", augs = augs)

In [ ]:
idx = 1
title = str(augs[idx])


i = -1
for main_class in classes:
    i += 1
    f, axs = plt.subplots(1, 3, figsize=(24, 6))
    f.suptitle(f"Noised by {title}", fontsize=16)
    key = f"{main_class} vs {main_class}"
    sns.kdeplot(mtd_of_clouds_clear[key], label=f"{main_class}", ax = axs[0], linestyle="--")
    sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=f"{main_class}", ax = axs[1], linestyle="--")
    sns.kdeplot(mtd_of_clouds_noised_noised[idx][key], label=f"{main_class}", ax = axs[2], linestyle="--")
    for second_class in classes:
        if second_class != main_class:
            key = f"{main_class} vs {second_class}"
            sns.kdeplot(mtd_of_clouds_clear[key], label=key, ax = axs[0])
            sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=key, ax = axs[1])
            sns.kdeplot(mtd_of_clouds_noised_noised[idx][key], label=key, ax = axs[2])

            axs[0].set_title('Pure-Pure')
            axs[1].set_title('Pure-Noise')
            axs[2].set_title('Noise-Noise')
            
    
    plt.legend(fontsize = "xx-small")

In [ ]:
i = -1
for main_class in classes:
    i += 1
    f, axs = plt.subplots(1, 4, figsize=(24, 6))
    f.suptitle(f"Comparing Pure-Noised with different degree", fontsize=16)
    key = f"{main_class} vs {main_class}"
    sns.kdeplot(mtd_of_clouds_clear[key], label=f"{main_class}", ax = axs[0], linestyle="--")
    for idx in range(len(augs)):
        sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=f"{main_class}", ax = axs[idx+1], linestyle="--")
        
    for second_class in classes:
        if second_class != main_class:
            key = f"{main_class} vs {second_class}"

            sns.kdeplot(mtd_of_clouds_clear[key], label=key, ax = axs[0])
            axs[0].set_title(f"Pure")
            
            for idx in range(len(augs)):
                sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=key, ax = axs[idx+1])
                title = str(augs[idx])
                axs[idx+1].set_title(f"Noised by {title}", fontsize=10)
            
    
    plt.legend(fontsize = "xx-small")

попробоваать искажения,  RTD autoencoder

## COIL-20

https://github.com/danchern97/RTD_AE/tree/main

In [ ]:
import os
from tqdm import tqdm
from PIL import Image
import numpy as np
import torch

filenames = os.listdir("COIL-20_data/coil-20-proc/")
dirname = "COIL-20_data/coil-20-proc/"

labels = []
data = []
for file in tqdm(filenames):
    img = Image.open(dirname + file)
    objId, imgId = file.split('__')
    imgId = int(imgId[:-4])
    objId = int(objId[3:])
    data.append(np.array(img))
    labels.append(objId)
data = np.asarray(data, dtype="float")
labels = np.asarray(labels)

In [ ]:
data.shape[0]/20

In [ ]:
classes = list(set(labels))

In [ ]:
img.show()

In [ ]:
augs = [AddGaussianNoise(0, 40), AddGaussianNoise(0, 80), AddGaussianNoise(0, 120)]

transformations = [transforms.Compose(
[transforms.ToTensor(), augmentation]) for augmentation in augs]

In [ ]:
data_noised = [np.array([np.array(trans(img)) for img in data]) for trans in transformations]
# data_noise = np.array([np.array(transform(img)) for img in data])

In [ ]:
fig = plt.figure(figsize=(8, 40)) 
gs = gridspec.GridSpec(len(classes), 4, width_ratios=[1,1,1,1], wspace=0.0, hspace=0.0)
color_map = 'gray'
for i in range(len(classes)):
    ax = plt.subplot(gs[i,0])
    if i == 0:
        ax.set_title(f"Pure", fontsize='xx-small')
    ax.imshow(data[labels==i+1][0].squeeze(), cmap = color_map)
    plt.xticks([])
    plt.yticks([])
    if i+1 in [15, 16, 17]: # classes with strange behaviour
        ax.set_ylabel(f"class number {i+1}", fontsize='xx-small', color="red")
    else:
        ax.set_ylabel(f"class number {i+1}", fontsize='xx-small')
    # ax.set_title(f"class number {i+1}", fontsize='xx-small')
    for j in range(3):
        ax = plt.subplot(gs[i,j+1])
        if i == 0:
            title = str(augs[j])
            ax.set_title(f"{title}", fontsize=5)
        ax.imshow(data_noised[j][labels==i+1][0].squeeze(), cmap = color_map)
        plt.xticks([])
        plt.yticks([])

In [ ]:
fig = plt.figure(figsize=(8, 40)) 
gs = gridspec.GridSpec(3, 4, width_ratios=[1,1,1,1], wspace=0.0, hspace=0.0)
color_map = 'gray'
for i in [15, 16, 17]:
    for j in np.

In [ ]:
from scipy.spatial.distance import cdist
from scipy.spatial import distance_matrix
class FurthestScaler:
    def __init__(self, p=2): # approximate
        self.is_fitted = False
        self.p = p
        
    def fit(self, data):
        self.furthest = self._furthest_distance(data)
        self.is_fitted = True
        
    def transform(self, data):
        if not self.is_fitted:
            raise NotFittedError
        return data / self.furthest
    
    def fit_transform(self, data):
        self.fit(data)
        return self.transform(data)

    def _furthest_distance(self, points, sample_frac=0.0):
        # exact solution, very computationaly expesive
        # hull = ConvexHull(points) 
        # hullpoints = points[hull.vertices,:]
        # hdist = distance_matrix(hullpoints, hullpoints, p=self.p)
        # approximation: upper bound
        # pick random point and compute distances to all of the points
        # diameter min: max(distances), diameter max (triangle inequality): 2 max(distances)
        if len(points.shape) > 2:
            points = points.reshape(points.shape[0],-1)
        idx = np.random.choice(np.arange(len(points)), size=1)
        hdist = distance_matrix(points[idx], points, p=self.p)
        return 0.1*hdist.max() # upper bound

In [ ]:
scaler = FurthestScaler()
data = torch.tensor(data).flatten(start_dim=1).numpy()
data = scaler.fit_transform(data)

data_noised =[torch.tensor(data_n).flatten(start_dim=1).numpy() for data_n in data_noised]
data_noised = [scaler.fit_transform(data_n) for data_n in data_noised]

In [ ]:
data.shape

In [ ]:
classes

In [ ]:
clouds = []
for i in classes:
    clouds.append(data[labels==i])

clouds_noised = [[], [], []]

for i in range(len(data_noised)):
    for j in classes:
        clouds_noised[i].append(data_noised[i][labels==j])

In [ ]:
# calculate_barcs_with_noise(clouds, clouds_noised, classes, "_COIL20", augs = augs, n_boostrap=100, batch_size1=50, batch_size2=60)


In [ ]:
mtd_of_clouds_clear, mtd_of_clouds_pure_noised, mtd_of_clouds_noised_noised = calculate_mtd_with_noise("_COIL20", augs = augs)

In [ ]:
idx = 1
title = str(augs[idx])


i = -1
for main_class in classes:
    i += 1
    f, axs = plt.subplots(1, 3, figsize=(24, 6))
    f.suptitle(f"Noised by {title}", fontsize=16)
    key = f"{main_class} vs {main_class}"
    sns.kdeplot(mtd_of_clouds_clear[key], label=f"{main_class}", ax = axs[0], linestyle="--")
    sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=f"{main_class}", ax = axs[1], linestyle="--")
    sns.kdeplot(mtd_of_clouds_noised_noised[idx][key], label=f"{main_class}", ax = axs[2], linestyle="--")
    for second_class in classes:
        if second_class != main_class:
            key = f"{main_class} vs {second_class}"
            sns.kdeplot(mtd_of_clouds_clear[key], label=key, ax = axs[0])
            sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=key, ax = axs[1])
            sns.kdeplot(mtd_of_clouds_noised_noised[idx][key], label=key, ax = axs[2])

            axs[0].set_title('Pure-Pure')
            axs[1].set_title('Pure-Noise')
            axs[2].set_title('Noise-Noise')
            
    
    plt.legend(fontsize = "xx-small")

In [ ]:
i = -1
for main_class in classes:
    i += 1
    f, axs = plt.subplots(1, 4, figsize=(24, 6))
    f.suptitle(f"Comparing Pure-Noised with different degree", fontsize=16)
    key = f"{main_class} vs {main_class}"
    sns.kdeplot(mtd_of_clouds_clear[key], label=f"{main_class}", ax = axs[0], linestyle="--")
    for idx in range(len(augs)):
        sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=f"{main_class}", ax = axs[idx+1], linestyle="--")
        
    for second_class in classes:
        if second_class != main_class:
            key = f"{main_class} vs {second_class}"

            sns.kdeplot(mtd_of_clouds_clear[key], label=key, ax = axs[0])
            axs[0].set_title(f"Pure")
            
            for idx in range(len(augs)):
                sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=key, ax = axs[idx+1])
                title = str(augs[idx])
                axs[idx+1].set_title(f"Noised by {title}", fontsize=10)
            
    
    plt.legend(fontsize = "xx-small")

добавить различные уровни шума, добавить тексты как облака точек (ембединговые представления для текстов) или другие области

imageNet

разные виды шума попробовать

---

### Let's take a closer look at text point clouds

In [ ]:
import pickle
with open("Data/cross_ripsnet_text_exp/human_gpt3_davinci_003_pds_10000_50_boostrap", 'rb') as fp:   #Pickling
    train_PD = pickle.load(fp)

In [ ]:
buffer = np.array([[i, j] for i in np.arange(100) for j in np.arange(100)])
# buffer_id =  np.random.choice(len(buffer), 2500, replace = False)
train_indexes = buffer

In [ ]:
train_indexes[101:110:2]

In [ ]:
train_indexes[100:110:2]

In [ ]:
classes = {"1":"GPT", "0": "Human"}

In [ ]:
len(train_PD[0][1][1])

1-gpt

0-human

In [ ]:
for cloud_samples in train_PD:
    for sample in cloud_samples:
        if len(sample[1]) == 0:
            sample[1] = np.array([[0., 0.]])

### I want to look at mtd density of gpt-gpt and gpt-human cross-barcodes at one figure

#### First GPT

In [ ]:
mtd_of_clouds_gpt_gpt_1 = []
mtd_of_clouds_gpt_human_1 = []

mtd_of_clouds_gpt_gpt_0 = []
mtd_of_clouds_gpt_human_0 = []

for value in train_PD[101:110:2]:
    mtd_of_clouds_gpt_gpt_1.append([mtd.get_score(barc, 1, 'sum_length') for barc in value])
    mtd_of_clouds_gpt_gpt_0.append([mtd.get_score(barc, 0, 'sum_length') for barc in value])

for value in train_PD[100:110:2]:
    mtd_of_clouds_gpt_human_1.append([mtd.get_score(barc, 1, 'sum_length') for barc in value])
    mtd_of_clouds_gpt_human_0.append([mtd.get_score(barc, 0, 'sum_length') for barc in value])

In [ ]:
f, axs = plt.subplots(1, 1, figsize=(24, 6))
f.suptitle(f"MTD density of gpt-gpt and gpt-human cross-barcodes H1", fontsize=16)
# key = f"{main_class} vs {main_class}"
for i in range(1,len(mtd_of_clouds_gpt_gpt_1)):
    sns.kdeplot(mtd_of_clouds_gpt_gpt_1[i], label="gpt vs gpt", ax = axs, linestyle="-")
    sns.kdeplot(mtd_of_clouds_gpt_human_1[i], label="gpt vs human", ax = axs, linestyle="--")
plt.legend(fontsize = "xx-small")
plt.show()

In [ ]:
f, axs = plt.subplots(1, 1, figsize=(24, 6))
f.suptitle(f"MTD density of gpt-gpt and gpt-human cross-barcodes H0", fontsize=16)
# key = f"{main_class} vs {main_class}"
for i in range(1,len(mtd_of_clouds_gpt_gpt_0)):
    sns.kdeplot(mtd_of_clouds_gpt_gpt_0[i], label="gpt vs gpt", ax = axs, linestyle="-")
    sns.kdeplot(mtd_of_clouds_gpt_human_0[i], label="gpt vs human", ax = axs, linestyle="--")
plt.legend(fontsize = "xx-small")
plt.show()

---

#### First Human

In [ ]:
mtd_of_clouds_human_gpt_1 = []
mtd_of_clouds_human_human_1 = []

mtd_of_clouds_human_gpt_0 = []
mtd_of_clouds_human_human_0 = []

for value in train_PD[1:10:2]:
    mtd_of_clouds_human_gpt_1.append([mtd.get_score(barc, 1, 'sum_length') for barc in value])
    mtd_of_clouds_human_gpt_0.append([mtd.get_score(barc, 0, 'sum_length') for barc in value])

for value in train_PD[0:10:2]:
    mtd_of_clouds_human_human_1.append([mtd.get_score(barc, 1, 'sum_length') for barc in value])
    mtd_of_clouds_human_human_0.append([mtd.get_score(barc, 0, 'sum_length') for barc in value])

In [ ]:
f, axs = plt.subplots(1, 1, figsize=(24, 6))
f.suptitle(f"MTD density of human-gpt and human-human cross-barcodes H1", fontsize=16)
# key = f"{main_class} vs {main_class}"
for i in range(1,len(mtd_of_clouds_human_gpt_1)):
    sns.kdeplot(mtd_of_clouds_human_gpt_1[i], label="human vs gpt", ax = axs, linestyle="-")
    sns.kdeplot(mtd_of_clouds_human_human_1[i], label="human vs human", ax = axs, linestyle="--")
plt.legend(fontsize = "xx-small")
plt.show()

In [ ]:
f, axs = plt.subplots(1, 1, figsize=(24, 6))
f.suptitle(f"MTD density of human-gpt and human-human cross-barcodes H0", fontsize=16)
# key = f"{main_class} vs {main_class}"
for i in range(1,len(mtd_of_clouds_human_gpt_0)):
    sns.kdeplot(mtd_of_clouds_human_gpt_0[i], label="gpt vs gpt", ax = axs, linestyle="-")
    sns.kdeplot(mtd_of_clouds_human_human_0[i], label="gpt vs human", ax = axs, linestyle="--")
plt.legend(fontsize = "xx-small")
plt.show()

# А может быть есть различия в растоянии между PI

### $H_1$

In [ ]:
train_PD_1 = [[x[1] if len(x[1])>0 else np.array([[0., 0.]]) for x in barcs ] for barcs in train_PD]

In [ ]:
n_boostrap = 50

In [ ]:
from gudhi.representations import DiagramSelector
pds_train = DiagramSelector(use=True).fit_transform(train_PD_1[0])
vpdtr = np.vstack(pds_train)

for barcs in train_PD_1[1:200]:
    pds_train = DiagramSelector(use=True).fit_transform(barcs)
    vpdtr = np.vstack((vpdtr, np.vstack(pds_train)))

pers = vpdtr[:,1]-vpdtr[:,0]
# bps_pairs = pairwise_distances(np.hstack([vpdtr[:,0:1],vpdtr[:,1:2]-vpdtr[:,0:1]])[:200]).flatten()
# ppers = bps_pairs[np.argwhere(bps_pairs > 1e-5).ravel()]
# sigma = np.quantile(ppers, .2) ## доделать
im_bnds = [np.min(vpdtr[:,0]), np.quantile(vpdtr[:,0],0.9), np.min(pers), np.max(pers)]

In [ ]:
from sklearn.metrics import pairwise_distances
from gudhi.representations import  PersistenceImage

PI_train_all = []
for barcs in tqdm(train_PD_1[:1000]):
    pds_train = DiagramSelector(use=True).fit_transform(barcs)
    pds_train = [item.astype(np.float32) for item in pds_train]
    # clean_pds_test = DiagramSelector(use=True).fit_transform(train_PD_1[800:])
    
    
    vpdtr = np.vstack(pds_train)
    pers = vpdtr[:,1]-vpdtr[:,0]
    bps_pairs = pairwise_distances(np.hstack([vpdtr[:,0:1],vpdtr[:,1:2]-vpdtr[:,0:1]])[:200]).flatten()
    ppers = bps_pairs[np.argwhere(bps_pairs > 1e-5).ravel()]
    sigma = np.quantile(ppers, .01)
    
    
    PI_params = {'bandwidth': sigma, 'weight': lambda x: x[1]**2, 
                 'resolution': [50,50], 'im_range': im_bnds}
    PI_train = PersistenceImage(**PI_params).fit_transform(pds_train)
    # clean_PI_test = PersistenceImage(**PI_params).fit_transform(clean_pds_test)
    PI_train = np.sum(PI_train,axis = 0)/n_boostrap
    PI_train = PI_train/np.sum(PI_train)
    PI_train_all.append(PI_train)
    
PI_train_all = np.array(PI_train_all)

In [ ]:
from utils import measure_dist

In [ ]:
fig = plt.figure(figsize=(8, 120)) 
gs = gridspec.GridSpec(40, 2, width_ratios=[1,1], wspace=2.0, hspace=0.0)
for i in range(40):
    idx = 20*i
    # P = [PI_train_all[idx]]
    # Q = [PI_train_all[idx+1]]
    ax = plt.subplot(gs[i, 0])
    ax.imshow(np.flip(np.reshape(PI_train_all[idx], [50,50]), 0), cmap='jet')
    plt.xticks([])
    plt.yticks([])
    ax.set_title(f"{classes[str(train_indexes[idx][0]%2)]} vs {classes[str(train_indexes[idx][1]%2)]}")
    indexes = train_indexes[idx]
    ax.set_ylabel(f"idx = {indexes}", fontsize='xx-small')
    
    ax = plt.subplot(gs[i, 1])
    ax.imshow(np.flip(np.reshape(PI_train_all[idx+1], [50,50]), 0), cmap='jet')
    plt.xticks([])
    plt.yticks([])
    ax.set_title(f"{classes[str(train_indexes[idx+1][0]%2)]} vs {classes[str(train_indexes[idx+1][1]%2)]}")
    indexes = train_indexes[idx+1]
    ax.set_ylabel(f"idx = {indexes}", fontsize='xx-small')

###  Создается ощущение, что картинки сгенерированные с левой частью из ГПТ более скудные, тонкие и менее разнообразны

In [ ]:
human_first = []
gpt_first = []
for i in tqdm(range(10)):
    distances = []
    for j in range(100):
        for k in range(100):
            if j!=k:
                P = [PI_train_all[i*100+j]]
                Q = [PI_train_all[i*100+k]]
                distances.append(measure_dist(P, Q, method="KL_sym"))
    if i%2 == 0:
        human_first.append(distances)
    else:
        gpt_first.append(distances)

In [ ]:
f, axs = plt.subplots(1, 1, figsize=(24, 6))
f.suptitle(f"Comparing densities of distances in Human_others and GPT_others PI collections", fontsize=16)
for i in tqdm(range(5)):
    sns.kdeplot(np.array(human_first[i]).flatten(), label="human", ax = axs, linestyle="-")
    sns.kdeplot(np.array(gpt_first[i]).flatten(), label="gpt", ax = axs, linestyle="--")
plt.legend(fontsize = "xx-small")
plt.show()

### We calculated inner distances for all pairs of texts grouped by the left cloud. In this picture, we marked where the left text was generated by a human or GPT.

### By these results, we can say that the Cross-PI differs by the author of left text.
---

$H_0$

In [ ]:
train_PD_0 = [[x[0] if len(x[0])>0 else np.array([[0., 0.]]) for x in barcs ] for barcs in train_PD]

In [ ]:
from gudhi.representations import DiagramSelector
pds_train = DiagramSelector(use=True).fit_transform(train_PD_1[0])
vpdtr = np.vstack(pds_train)

for barcs in train_PD_0[1:200]:
    pds_train = DiagramSelector(use=True).fit_transform(barcs)
    vpdtr = np.vstack((vpdtr, np.vstack(pds_train)))

pers = vpdtr[:,1]-vpdtr[:,0]
# bps_pairs = pairwise_distances(np.hstack([vpdtr[:,0:1],vpdtr[:,1:2]-vpdtr[:,0:1]])[:200]).flatten()
# ppers = bps_pairs[np.argwhere(bps_pairs > 1e-5).ravel()]
# sigma = np.quantile(ppers, .2) ## доделать
im_bnds = [np.min(vpdtr[:,0]),0.3, np.min(pers), np.max(pers)]

In [ ]:
from sklearn.metrics import pairwise_distances
from gudhi.representations import  PersistenceImage

PI_train_all_0 = []
for barcs in tqdm(train_PD_0[:1000]):
    pds_train = DiagramSelector(use=True).fit_transform(barcs)
    pds_train = [item.astype(np.float32) for item in pds_train]
    # clean_pds_test = DiagramSelector(use=True).fit_transform(train_PD_1[800:])
    
    
    vpdtr = np.vstack(pds_train)
    pers = vpdtr[:,1]-vpdtr[:,0]
    bps_pairs = pairwise_distances(np.hstack([vpdtr[:,0:1],vpdtr[:,1:2]-vpdtr[:,0:1]])[:200]).flatten()
    ppers = bps_pairs[np.argwhere(bps_pairs > 1e-5).ravel()]
    sigma = np.quantile(ppers, .02)
    
    
    PI_params = {'bandwidth': sigma, 'weight': lambda x: x[1]**2, 
                 'resolution': [50,50], 'im_range': im_bnds}
    PI_train = PersistenceImage(**PI_params).fit_transform(pds_train)
    # clean_PI_test = PersistenceImage(**PI_params).fit_transform(clean_pds_test)
    PI_train = np.sum(PI_train,axis = 0)/n_boostrap
    PI_train = PI_train/np.sum(PI_train)
    PI_train_all_0.append(PI_train)
    
PI_train_all_0 = np.array(PI_train_all_0)

In [ ]:
fig = plt.figure(figsize=(8, 120)) 
gs = gridspec.GridSpec(40, 2, width_ratios=[1,1], wspace=2.0, hspace=0.0)
for i in range(40):
    idx = 20*i
    # P = [PI_train_all[idx]]
    # Q = [PI_train_all[idx+1]]
    ax = plt.subplot(gs[i, 0])
    ax.imshow(np.flip(np.reshape(PI_train_all_0[idx], [50,50]), 0), cmap='jet')
    plt.xticks([])
    plt.yticks([])
    ax.set_title(f"{classes[str(train_indexes[idx][0]%2)]} vs {classes[str(train_indexes[idx][1]%2)]}")
    indexes = train_indexes[idx]
    ax.set_ylabel(f"idx = {indexes}", fontsize='xx-small')
    
    ax = plt.subplot(gs[i, 1])
    ax.imshow(np.flip(np.reshape(PI_train_all_0[idx+1], [50,50]), 0), cmap='jet')
    plt.xticks([])
    plt.yticks([])
    ax.set_title(f"{classes[str(train_indexes[idx+1][0]%2)]} vs {classes[str(train_indexes[idx+1][1]%2)]}")
    indexes = train_indexes[idx+1]
    ax.set_ylabel(f"idx = {indexes}", fontsize='xx-small')

In [ ]:
human_first_0 = []
gpt_first_0 = []
for i in tqdm(range(10)):
    distances = []
    for j in range(100):
        for k in range(100):
            if j!=k:
                P = [PI_train_all_0[i*100+j]]
                Q = [PI_train_all_0[i*100+k]]
                distances.append(measure_dist(P, Q, method="KL"))
    if i%2 == 0:
        human_first_0.append(distances)
    else:
        gpt_first_0.append(distances)

In [ ]:
f, axs = plt.subplots(1, 1, figsize=(24, 6))
f.suptitle(f"Comparing densities of distances in Human_others and GPT_others PI collections H0", fontsize=16)
for i in tqdm(range(5)):
    sns.kdeplot(np.array(human_first_0[i]).flatten(), label="human", ax = axs, linestyle="-")
    sns.kdeplot(np.array(gpt_first_0[i]).flatten(), label="gpt", ax = axs, linestyle="--")
plt.legend(fontsize = "xx-small")
plt.show()

### Lets use full dataset and try to visualize PI in 2d, maybe we will find smth!

In [ ]:
import pickle
with open("Data/cross_ripsnet_text_exp/human_gpt3_davinci_003_PI_10000_50_boostrap", 'rb') as fp:   #Pickling
    PI_train_all = pickle.load(fp)
PI_train_all.shape

In [ ]:
PI_pictures = [np.flip(np.reshape(pic, [50,50]), 0) for pic in PI_train_all]

In [ ]:
buffer = np.array([[i, j] for i in np.arange(100) for j in np.arange(100)])
train_indexes = buffer

labels_of_left_clouds = [classes[str(train_indexes[idx][0]%2)] for idx in range(len(train_indexes))]

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.manifold import Isomap
from sklearn.manifold import LocallyLinearEmbedding


pca = PCA(n_components=2)
two_d_pics_pca = pca.fit_transform(PI_train_all)

isomap = Isomap(n_components=2)
two_d_pics_isomap = isomap.fit_transform(PI_train_all)

tsne = TSNE(n_components=2)
two_d_pics_tsne = tsne.fit_transform(PI_train_all)

lle = LocallyLinearEmbedding(n_components=2)
two_d_pics_lle = lle.fit_transform(PI_train_all)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

def plot_dimreduction_result(data, labels, all_pictures, title, plot_pics = True):
    X = MinMaxScaler().fit_transform(data)
    _, ax = plt.subplots()
    sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=True, ax = ax,alpha = 0.7)
    if plot_pics:
        shown_images = np.array([[1.0, 1.0]])  # just something big
        for i in range(X.shape[0]):
            # plot every digit on the embedding
            # show an annotation box for a group of digits
            dist = np.sum((X[i] - shown_images) ** 2, 1)
            if np.min(dist) < 4e-3:
                # don't show points that are too close
                continue
            shown_images = np.concatenate([shown_images, [X[i]]], axis=0)
            imagebox = offsetbox.AnnotationBbox(
                offsetbox.OffsetImage(all_pictures[i],zoom = 0.3, cmap='jet'), X[i], pad = 0.1
            )
            imagebox.set(zorder=1)
            ax.add_artist(imagebox)

    ax.set_title(title)
    ax.axis("off")
    # plt.scatter(two_d_pics_isomap [:,0], two_d_pics_isomap [:,1],)
    # plt.title("Isomap")
    plt.show()


plot_dimreduction_result(two_d_pics_pca, labels_of_left_clouds, PI_pictures, title = "PCA", plot_pics=False)

plot_dimreduction_result(two_d_pics_isomap, labels_of_left_clouds, PI_pictures, title = "Isomap", plot_pics=False)

plot_dimreduction_result(two_d_pics_tsne, labels_of_left_clouds, PI_pictures, title = "TSNE", plot_pics=False)

plot_dimreduction_result(two_d_pics_lle, labels_of_left_clouds, PI_pictures, title = "LocallyLinearEmbedding", plot_pics=False)


---

### Let's add CIFAR100 with much more clouds to our experiments

In [ ]:
augs = [AddGaussianNoise(0, 0.05), AddGaussianNoise(0, 0.1), AddGaussianNoise(0, 0.2)]

transformations = [transforms.Compose(
[transforms.ToTensor(), augmentation]) for augmentation in augs]

trainset = torchvision.datasets.CIFAR100(root='./data_cifar100', train=True,
                                        download=True, transform=transforms.ToTensor())


trainset_noised = [torchvision.datasets.CIFAR100(root='./data_cifar100', train=True,
                                        download=True, transform=trans) for trans in transformations]


In [ ]:
len(trainset)

In [ ]:
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

imshow(trainset[0][0])
imshow(trainset_noised[0][0][0])
imshow(trainset_noised[1][0][0])
imshow(trainset_noised[2][0][0])

In [ ]:
classes = trainset.classes

In [ ]:
batch_size = 50000

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

dataiter = iter(trainloader)
images, labels = next(dataiter)

images = images.numpy().mean(axis = 1)

images = images.reshape(50000,-1)

images.shape

In [ ]:
clouds = []
for i in range(len(classes)):
    clouds.append(images[labels==i])

In [ ]:
batch_size = 50000
clouds_noised = [[], [], []]

for i in range(len(trainset_noised)):
    trainloader = torch.utils.data.DataLoader(trainset_noised[i], batch_size=batch_size,
                                              shuffle=True, num_workers=1)

    dataiter = iter(trainloader)
    images, labels = next(dataiter)
    
    images = images.numpy().mean(axis = 1)
    
    images = images.reshape(50000,-1)
    
    for j in range(len(classes)):
        clouds_noised[i].append(images[labels==j])

images.shape

In [ ]:
clouds_noised[0][0].shape

---

In [ ]:
calculate_barcs_with_noise(clouds, clouds_noised, classes, "_CIFAR100", augs = augs, n_boostrap=30, batch_size1=200, batch_size2=300)


In [ ]:
mtd_of_clouds_clear, mtd_of_clouds_pure_noised, mtd_of_clouds_noised_noised = calculate_mtd_with_noise("_CIFAR100", augs = augs)

In [ ]:
idx = 1
title = str(augs[idx])


i = -1
for main_class in classes[:10]:
    i += 1
    f, axs = plt.subplots(1, 3, figsize=(24, 6))
    f.suptitle(f"Noised by {title}", fontsize=16)
    key = f"{main_class} vs {main_class}"
    sns.kdeplot(mtd_of_clouds_clear[key], label=f"{main_class}", ax = axs[0], linestyle="--")
    sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=f"{main_class}", ax = axs[1], linestyle="--")
    sns.kdeplot(mtd_of_clouds_noised_noised[idx][key], label=f"{main_class}", ax = axs[2], linestyle="--")
    for second_class in classes:
        if second_class != main_class:
            key = f"{main_class} vs {second_class}"
            sns.kdeplot(mtd_of_clouds_clear[key], label=key, ax = axs[0])
            sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=key, ax = axs[1])
            sns.kdeplot(mtd_of_clouds_noised_noised[idx][key], label=key, ax = axs[2])

            axs[0].set_title('Pure-Pure')
            axs[1].set_title('Pure-Noise')
            axs[2].set_title('Noise-Noise')
            
    
    # plt.legend(fontsize = "xx-small")

In [ ]:
i = -1
for main_class in classes[:10]:
    i += 1
    f, axs = plt.subplots(1, 4, figsize=(24, 6))
    f.suptitle(f"Comparing Pure-Noised with different degree", fontsize=16)
    key = f"{main_class} vs {main_class}"
    sns.kdeplot(mtd_of_clouds_clear[key], label=f"{main_class}", ax = axs[0], linestyle="--")
    for idx in range(len(augs)):
        sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=f"{main_class}", ax = axs[idx+1], linestyle="--")
        
    for second_class in classes:
        if second_class != main_class:
            key = f"{main_class} vs {second_class}"

            sns.kdeplot(mtd_of_clouds_clear[key], label=key, ax = axs[0])
            axs[0].set_title(f"Pure")
            
            for idx in range(len(augs)):
                sns.kdeplot(mtd_of_clouds_pure_noised[idx][key], label=key, ax = axs[idx+1])
                title = str(augs[idx])
                axs[idx+1].set_title(f"Noised by {title}", fontsize=10)
            
    
    # plt.legend(fontsize = "xx-small")

Интересно то что дисперсия с добавлением шума сильно увеличивается. В сложных случаях, когда добавление шума не помогает отделить нужное облако, мы можем использовать различие в дисперсии.
